### Rolling statistics and moving averages are statistical methods used to summarize information over a specific period of time. One such method is the 7-day rolling mean.

In [3]:
import pandas as pd
from datetime import datetime, timedelta

# Importing data
df = pd.read_csv(r"E:\Projects\Python\Predicting_Energy_Consumption_dataset\london_energy.csv")

# Grouping the records per date in order to calculate the average consumption over all households.
df_avg_consumption = df.groupby("Date")["KWH"].mean()
df_avg_consumption = pd.DataFrame({"date": df_avg_consumption.index.tolist(), "consumption": df_avg_consumption.values.tolist()})
df_avg_consumption["date"] = pd.to_datetime(df_avg_consumption["date"])

# Enhancing the data with date-related features
df_avg_consumption["day_of_week"] = df_avg_consumption["date"].dt.dayofweek
df_avg_consumption["day_of_year"] = df_avg_consumption["date"].dt.dayofyear
df_avg_consumption["month"] = df_avg_consumption["date"].dt.month
df_avg_consumption["quarter"] = df_avg_consumption["date"].dt.quarter
df_avg_consumption["year"] = df_avg_consumption["date"].dt.year

# Adding weather features
df_weather = pd.read_csv(r"E:\Projects\Python\Predicting_Energy_Consumption_dataset\london_weather.csv")

# Parsing dates
df_weather["date"] = pd.to_datetime(df_weather["date"], format="%Y%m%d")

# Filling missing values through interpolation
df_weather = df_weather.interpolate(method="ffill")

# Enhancing consumption dataset with weather information
df_avg_consumption = df_avg_consumption.merge(df_weather, how="inner", on="date")

C:\Users\rahul\AppData\Local\Temp\ipykernel_16904\2909999023.py:26: FutureWarning: DataFrame.interpolate with method=ffill is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_weather = df_weather.interpolate(method="ffill")


In [4]:
# Rolling Mean in conjuction wiht XGBoost
# Choose which window works best
df_avg_consumption['rolling_mean_14'] = df_avg_consumption['consumption'].rolling(window=14).mean().shift()
df_avg_consumption['rolling_mean_7'] = df_avg_consumption['consumption'].rolling(window=7).mean().shift()
df_avg_consumption['rolling_mean_4'] = df_avg_consumption['consumption'].rolling(window=4).mean().shift()
df_avg_consumption['rolling_mean_2'] = df_avg_consumption['consumption'].rolling(window=2).mean().shift()


In [ ]:
# Create lag features for past consumption values
lags = [2, 4, 7, 9, 14]  # Define the lag periods

for lag in lags:
    df_avg_consumption[f"lag_{lag}"] = df_avg_consumption["consumption"].shift(lag)

# Drop NaN values after shifting
df_avg_consumption = df_avg_consumption.dropna()

# Splitting Training/Testing Data
training_mask = df_avg_consumption["date"] < "2013-07-28"
training_data = df_avg_consumption.loc[training_mask]
testing_mask = df_avg_consumption["date"] >= "2013-07-28"
testing_data = df_avg_consumption.loc[testing_mask]

# Dropping unnecessary `date` column
training_data = training_data.drop(columns=["date"])
testing_dates = testing_data["date"]
testing_data = testing_data.drop(columns=["date"])

# Define feature set
features = ["day_of_week", "day_of_year", "month", "quarter", "year",
            "cloud_cover", "sunshine", "global_radiation", "max_temp",
            "mean_temp", "min_temp", "precipitation", "pressure",
            "snow_depth", "rolling_mean_2"] + [f"lag_{lag}" for lag in lags]

X_train = training_data[features]
y_train = training_data["consumption"]

X_test = testing_data[features]
y_test = testing_data["consumption"]

# XGBoost training
cv_split = TimeSeriesSplit(n_splits=4, test_size=100)
model = XGBRegressor()
parameters = {
    "max_depth": [3, 4, 6, 5, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "n_estimators": [100, 300, 500, 700, 900, 1000],
    "colsample_bytree": [0.3, 0.5, 0.7]
}

grid_search = GridSearchCV(estimator=model, cv=cv_split, param_grid=parameters)
grid_search.fit(X_train, y_train)

def evaluate_model(y_test, prediction):
    print(f"MAE: {mean_absolute_error(y_test, prediction)}")
    print(f"MSE: {mean_squared_error(y_test, prediction)}")
    print(f"MAPE: {mean_absolute_percentage_error(y_test, prediction)}")
  
# Evaluating GridSearch results
prediction = grid_search.predict(X_test)
evaluate_model(y_test, prediction)


MAE: 0.2623338098704692
MSE: 0.577822909785803
MAPE: 0.13569298933459276
